In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

file_path = "/content/drive/MyDrive/NLP_final_project/news_cleaned_notebook1.parquet"

df = pd.read_parquet(file_path, engine="pyarrow")
print(df.shape)
df.head()

(199380, 9)


,url,date,language,title,text,full_text,title_len,text_len,full_text_len
0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...",77,3500,3578
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,78,4820,4899
2,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...",54,5123,5178
3,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,en,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,102,4017,4120
4,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,en,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,65,4290,4356


In [3]:
import re

def normalize_for_match(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text_for_match"] = df["full_text"].apply(normalize_for_match)

In [4]:
AI_CORE = [
    "artificial intelligence", "ai", "machine learning", "ml",
    "generative ai", "gen ai", "llm", "large language model",
    "deep learning", "neural network", "computer vision",
    "natural language processing", "nlp", "foundation model",
    "chatgpt", "gpt", "openai", "anthropic", "gemini", "claude",
    "copilot", "automation", "automated"
]

IMPACT_TERMS = [
    "productivity", "efficiency", "cost reduction", "cost savings",
    "workflow", "workflow redesign", "job loss", "layoff", "layoffs",
    "displacement", "augment", "augmentation", "replace workers",
    "replace jobs", "transform", "disrupt", "adoption", "deployment",
    "implementation", "investment", "roi", "benefit", "benefits",
    "risk", "risks", "use case", "labor", "workforce"
]

BUSINESS_TERMS = [
    "industry", "industries", "company", "companies", "enterprise",
    "business", "sector", "market", "firm", "organization",
    "operations", "employee", "employees", "employer"
]

INDUSTRY_HINTS = [
    "healthcare", "finance", "banking", "insurance", "legal",
    "manufacturing", "retail", "education", "logistics", "media",
    "marketing", "consulting", "construction", "transportation",
    "pharmaceutical", "telecom", "energy", "government"
]


In [5]:
def count_keyword_hits(text, keywords):
    count = 0
    for kw in keywords:
        pattern = r"\b" + re.escape(kw.lower()) + r"\b"
        if re.search(pattern, text):
            count += 1
    return count

df["ai_core_hits"] = df["text_for_match"].apply(lambda x: count_keyword_hits(x, AI_CORE))
df["impact_hits"] = df["text_for_match"].apply(lambda x: count_keyword_hits(x, IMPACT_TERMS))
df["business_hits"] = df["text_for_match"].apply(lambda x: count_keyword_hits(x, BUSINESS_TERMS))
df["industry_hits"] = df["text_for_match"].apply(lambda x: count_keyword_hits(x, INDUSTRY_HINTS))

df["relevance_score"] = (
    3 * df["ai_core_hits"] +
    2 * df["impact_hits"] +
    1 * df["business_hits"] +
    1 * df["industry_hits"]
)

df[["ai_core_hits", "impact_hits", "business_hits", "industry_hits", "relevance_score"]].head()

,ai_core_hits,impact_hits,business_hits,industry_hits,relevance_score
0,1,2,1,0,8
1,1,0,1,0,4
2,6,0,0,1,19
3,4,3,0,1,19
4,3,1,0,0,11


In [6]:
df["has_ai_signal"] = df["ai_core_hits"] > 0

df["is_relevant_rule"] = (
    (df["has_ai_signal"]) &
    (
        (df["impact_hits"] >= 1) |
        (df["business_hits"] >= 1) |
        (df["industry_hits"] >= 1) |
        (df["relevance_score"] >= 4)
    )
)

df["is_relevant_rule"].value_counts(dropna=False)

,count
is_relevant_rule,
True,190293
False,9087


In [7]:
df[df["is_relevant_rule"]][
    ["date", "title", "relevance_score", "ai_core_hits", "impact_hits", "business_hits", "industry_hits"]
].sample(15, random_state=42)

,date,title,relevance_score,ai_core_hits,impact_hits,business_hits,industry_hits
76214,2025-08-29,"Video: Google Spam Update, AI Mode Changes, Ch...",12,2,0,4,2
94940,2024-09-17,It’s happening: AMD is going to use AI in FSR ...,14,3,1,2,1
95682,2023-08-05,ChatGPT data analysis performance tested - Gee...,14,4,0,1,1
33419,2023-07-26,"Google, Microsoft, OpenAI and Anthropic announ...",24,4,3,5,1
120352,2023-05-31,Ordaōs Selected to Participate in the AWS Gene...,16,3,0,5,2
186972,2023-01-06,AI writing programs: Artificial intelligence c...,14,3,1,3,0
157677,2023-01-17,Sapia.ai's new research reinforces its ethical...,23,6,1,2,1
25379,2023-05-18,"As Tom Hanks sparks debate on AI in films, che...",18,4,1,2,2
65881,2023-06-13,Artificial Intelligence picks the most beautif...,12,2,1,2,2
1062,2024-04-04,"Israel used AI to identify 37,000 targets asso...",21,4,2,3,2


In [8]:
df[~df["is_relevant_rule"]][
    ["date", "title", "relevance_score", "ai_core_hits", "impact_hits", "business_hits", "industry_hits"]
].sample(15, random_state=42)

,date,title,relevance_score,ai_core_hits,impact_hits,business_hits,industry_hits
97954,2024-02-12,Standing portrait outdoors person. AI | Free P...,3,1,0,0,0
72163,2022-01-06,Pango Group and Pango Foundation Share 2021 Ph...,7,0,0,4,3
92876,2025-06-01,Aerojet Rocketdyne Data Scientist Salary | $74...,6,0,1,4,0
173491,2024-08-07,"In 2025, construction work on the second stage...",11,0,1,5,4
167783,2025-11-20,The sad thing about artificial intelligence is...,3,1,0,0,0
86631,2022-08-04,Frontiers in Communications and Networks | Dat...,1,0,0,0,1
150714,2023-10-01,DeepFiction AI - Desktop App for Mac and PC - ...,3,1,0,0,0
177225,2023-06-09,U.S.-U.K. plan would strengthen cooperation fr...,3,1,0,0,0
156169,2024-02-14,Dragon moon outdoors nature. AI | Free Photo -...,3,1,0,0,0
186541,2025-01-25,Everlaw Data Scientist Salary | $125K-$177K+ |...,6,0,1,4,0


In [9]:
df[df["is_relevant_rule"]][
    ["date", "title", "text", "relevance_score"]
].sample(10, random_state=42)

,date,title,text,relevance_score
76214,2025-08-29,"Video: Google Spam Update, AI Mode Changes, Ch...","Video: Google Spam Update, AI Mode Changes, Ch...",12
94940,2024-09-17,It’s happening: AMD is going to use AI in FSR ...,It’s happening: AMD is going to use AI in FSR ...,14
95682,2023-08-05,ChatGPT data analysis performance tested - Gee...,ChatGPT data analysis performance tested - Gee...,14
33419,2023-07-26,"Google, Microsoft, OpenAI and Anthropic announ...","Google, Microsoft, OpenAI and Anthropic announ...",24
120352,2023-05-31,Ordaōs Selected to Participate in the AWS Gene...,Ordaōs Selected to Participate in the AWS Gene...,16
186972,2023-01-06,AI writing programs: Artificial intelligence c...,AI writing programs: Artificial intelligence c...,14
157677,2023-01-17,Sapia.ai's new research reinforces its ethical...,Sapia.ai's new research reinforces its ethical...,23
25379,2023-05-18,"As Tom Hanks sparks debate on AI in films, che...","As Tom Hanks sparks debate on AI in films, che...",18
65881,2023-06-13,Artificial Intelligence picks the most beautif...,Artificial Intelligence picks the most beautif...,12
1062,2024-04-04,"Israel used AI to identify 37,000 targets asso...","Israel used AI to identify 37,000 targets asso...",21


In [10]:
df[~df["is_relevant_rule"]][
    ["date", "title", "text", "relevance_score"]
].sample(10, random_state=42)

,date,title,text,relevance_score
97954,2024-02-12,Standing portrait outdoors person. AI | Free P...,Standing portrait outdoors person. AI | Free P...,3
72163,2022-01-06,Pango Group and Pango Foundation Share 2021 Ph...,Pango Group and Pango Foundation Share 2021 Ph...,7
92876,2025-06-01,Aerojet Rocketdyne Data Scientist Salary | $74...,Aerojet Rocketdyne Data Scientist Salary | $74...,6
173491,2024-08-07,"In 2025, construction work on the second stage...","In 2025, construction work on the second stage...",11
167783,2025-11-20,The sad thing about artificial intelligence is...,The sad thing about artificial intelligence is...,3
86631,2022-08-04,Frontiers in Communications and Networks | Dat...,Frontiers in Communications and Networks | Dat...,1
150714,2023-10-01,DeepFiction AI - Desktop App for Mac and PC - ...,DeepFiction AI - Desktop App for Mac and PC - ...,3
177225,2023-06-09,U.S.-U.K. plan would strengthen cooperation fr...,U.S.-U.K. plan would strengthen cooperation fr...,3
156169,2024-02-14,Dragon moon outdoors nature. AI | Free Photo -...,Dragon moon outdoors nature. AI | Free Photo -...,3
186541,2025-01-25,Everlaw Data Scientist Salary | $125K-$177K+ |...,Everlaw Data Scientist Salary | $125K-$177K+ |...,6


In [11]:
temp_keep = df[df["is_relevant_rule"]][
    ["date", "title", "text", "relevance_score"]
].sample(10, random_state=42).copy()

temp_keep["text_preview"] = temp_keep["text"].str[:300]
temp_keep[["date", "title", "relevance_score", "text_preview"]]

,date,title,relevance_score,text_preview
76214,2025-08-29,"Video: Google Spam Update, AI Mode Changes, Ch...",12,"Video: Google Spam Update, AI Mode Changes, Ch..."
94940,2024-09-17,It’s happening: AMD is going to use AI in FSR ...,14,It’s happening: AMD is going to use AI in FSR ...
95682,2023-08-05,ChatGPT data analysis performance tested - Gee...,14,ChatGPT data analysis performance tested - Gee...
33419,2023-07-26,"Google, Microsoft, OpenAI and Anthropic announ...",24,"Google, Microsoft, OpenAI and Anthropic announ..."
120352,2023-05-31,Ordaōs Selected to Participate in the AWS Gene...,16,Ordaōs Selected to Participate in the AWS Gene...
186972,2023-01-06,AI writing programs: Artificial intelligence c...,14,AI writing programs: Artificial intelligence c...
157677,2023-01-17,Sapia.ai's new research reinforces its ethical...,23,Sapia.ai's new research reinforces its ethical...
25379,2023-05-18,"As Tom Hanks sparks debate on AI in films, che...",18,"As Tom Hanks sparks debate on AI in films, che..."
65881,2023-06-13,Artificial Intelligence picks the most beautif...,12,Artificial Intelligence picks the most beautif...
1062,2024-04-04,"Israel used AI to identify 37,000 targets asso...",21,"Israel used AI to identify 37,000 targets asso..."


In [12]:
temp_drop = df[~df["is_relevant_rule"]][
    ["date", "title", "text", "relevance_score"]
].sample(10, random_state=42).copy()

temp_drop["text_preview"] = temp_drop["text"].str[:300]
temp_drop[["date", "title", "relevance_score", "text_preview"]]

,date,title,relevance_score,text_preview
97954,2024-02-12,Standing portrait outdoors person. AI | Free P...,3,Standing portrait outdoors person. AI | Free P...
72163,2022-01-06,Pango Group and Pango Foundation Share 2021 Ph...,7,Pango Group and Pango Foundation Share 2021 Ph...
92876,2025-06-01,Aerojet Rocketdyne Data Scientist Salary | $74...,6,Aerojet Rocketdyne Data Scientist Salary | $74...
173491,2024-08-07,"In 2025, construction work on the second stage...",11,"In 2025, construction work on the second stage..."
167783,2025-11-20,The sad thing about artificial intelligence is...,3,The sad thing about artificial intelligence is...
86631,2022-08-04,Frontiers in Communications and Networks | Dat...,1,Frontiers in Communications and Networks | Dat...
150714,2023-10-01,DeepFiction AI - Desktop App for Mac and PC - ...,3,DeepFiction AI - Desktop App for Mac and PC - ...
177225,2023-06-09,U.S.-U.K. plan would strengthen cooperation fr...,3,U.S.-U.K. plan would strengthen cooperation fr...
156169,2024-02-14,Dragon moon outdoors nature. AI | Free Photo -...,3,Dragon moon outdoors nature. AI | Free Photo -...
186541,2025-01-25,Everlaw Data Scientist Salary | $125K-$177K+ |...,6,Everlaw Data Scientist Salary | $125K-$177K+ |...


In [13]:
EXCLUDE_TERMS = [
    "price today", "market cap", "coin", "token", "crypto", "cryptocurrency",
    "stock price", "buy now", "watch video", "viral video", "photo gallery",
    "celebrity", "football", "basketball", "baseball", "match preview",
    "lottery", "horoscope"
]

def count_keyword_hits(text, keywords):
    count = 0
    for kw in keywords:
        pattern = r"\b" + re.escape(kw.lower()) + r"\b"
        if re.search(pattern, text):
            count += 1
    return count

df["exclude_hits"] = df["text_for_match"].apply(lambda x: count_keyword_hits(x, EXCLUDE_TERMS))

In [14]:
df["is_relevant"] = (
    df["is_relevant_rule"] &
    (df["exclude_hits"] == 0)
)

df["is_relevant"].value_counts(dropna=False)

,count
is_relevant,
True,129326
False,70054


In [15]:
df[df["is_relevant"]][
    ["date", "title", "relevance_score", "exclude_hits"]
].sample(15, random_state=42)

,date,title,relevance_score,exclude_hits
137955,2025-08-12,Alida Launches Industry-First AI Assistant for...,8,0
190167,2023-03-12,Suhel Seth on Future Impact of Artificial Inte...,12,0
96939,2022-08-15,ASTERRA's new satellite based PolSAR technolog...,24,0
6687,2024-01-04,Where federal AI is headed next | Federal News...,22,0
87385,2023-04-05,Exotel launches AI chatbot builder ‘ExoMind’ p...,33,0
42260,2022-11-09,Subtle Medical Joins Imaging AI in Practice De...,20,0
34744,2023-02-28,Wadhwani AI showcases Solutions for Social Goo...,17,0
118049,2023-04-13,Zoom and CrowdStrike stocks ride SaaS’s AI tre...,27,0
24378,2022-02-10,Venzee AI Accelerates Intelligent Data Syndica...,24,0
16312,2022-07-07,This AI will tell you whether you're being pai...,25,0


In [16]:
df[~df["is_relevant"]][
    ["date", "title", "relevance_score", "exclude_hits"]
].sample(15, random_state=42)

,date,title,relevance_score,exclude_hits
80031,2025-09-29,"Anthropic Launches Claude Sonnet 4.5, Introduc...",38,2
58899,2025-06-24,"Arrive AI Forges International Partnership, Br...",40,1
132383,2022-06-23,"How Much Amazon Pay Engineers, Analysts, Data ...",16,0
138150,2023-06-20,"ChatGPT, the credentials of over 100,000 compr...",9,1
122919,2025-10-13,Solana und Dogecoin sind Ablenkungen: AI nennt...,3,2
5908,2025-06-11,83 Inch LG OLED AI B5 4K Smart TV - OLED83B56L...,27,1
188794,2022-01-20,Siddhartha college establishes Centre of Excel...,23,1
195390,2024-02-12,Flower blossom petal plant. AI | Free Photo - ...,3,0
41640,2023-05-03,Scientists warn of AI dangers but don't agree ...,27,1
179433,2025-04-03,Elon Musk’s Grok AI on X Falsely Claims Val Ki...,5,1


In [17]:
df_relevant = df[df["is_relevant"]].copy()

print("Original shape:", df.shape)
print("Relevant shape:", df_relevant.shape)
print("Retention rate:", round(len(df_relevant) / len(df) * 100, 2), "%")

Original shape: (199380, 19)
Relevant shape: (129326, 19)
Retention rate: 64.86 %


In [18]:
df_relevant.to_parquet(
    "/content/drive/MyDrive/NLP_final_project/news_relevant_notebook2.parquet",
    engine="pyarrow",
    index=False
)

print("saved relevance-filtered file")

saved relevance-filtered file
